# GEIGER-1911 · 1 — Two atoms, and the one number that separates them

An atom is neutral, and it holds electrons. Where is the positive charge that balances them?

Two answers were on the table in 1910, and both fit everything then known.

* **The diffuse atom.** The positive charge is spread evenly through the whole atom, a
  ball of radius about $1.35 \times 10^{-10}$ m with the electrons embedded in it. A
  fast alpha particle crossing it is barely deflected, because at any point inside the
  ball it feels only the charge beneath it.
* **The hard centre.** The positive charge is concentrated in something very much
  smaller, and the alpha that comes close to it feels the whole of it at once.

They are not two models. They are **one model at two values of one number** — the radius
$R$ of the positive charge — and that is what makes this a design problem instead of a
debate. This notebook builds that model, in one expression tree, and works out what the
apparatus can and cannot see before any of it is switched on.

| | | reaches into |
|---|---|---|
| **1** | the number that separates them | `core.BASES`, `core.Dimension` |
| **2** | one mean function, both atoms | `core.Add`, `core.Apply`, `core.Pow`, `core.ModelSpec` |
| **3** | what the apparatus can never see | arithmetic, and it is the most important cell here |
| **4** | counting is Poisson; the design math is Gaussian | `design.fisher_information` |

In [ ]:
import sys

sys.path.insert(0, ".")

import math

import numpy as np
import plotly.graph_objects as go

import scattering as S
from axiom.core import D, ModelSpec, dimension, free_parameters, params
from axiom.design import fisher_information

from axiom.display import enable, table

sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import CRITICAL, MUTED, annotate, caption, curve_band, mark_y, scatter_fit, shade

enable();  # every axiom result renders itself from here on

print(f"alpha energy            {S.E_ALPHA} MeV (radium C')")
print(f"foil                    gold, {S.FOIL_THICKNESS * 1e6:.1f} um, "
      f"{S.AREAL_DENSITY:.3g} atoms/m2")
print(f"beam                    {S.BEAM_RATE:,.0f} alphas/s")
print(f"atoms crossed per alpha {S.n_encounters():,.0f}")

## 1 · The number that separates them

An alpha of charge $2e$ fired straight at a charge $79e$ stops and turns around where its
kinetic energy has all become potential energy:

$$D = \frac{2 \cdot 2 \cdot 79 \, e^2}{4\pi\varepsilon_0 E}$$

$D$ is the smallest distance in the experiment. Nothing in the beam gets closer to the
charge than that, at any angle, ever. A trajectory scattered through $\theta$ has its
closest approach at

$$r_\text{min}(\theta) = \frac{D}{2}\left(1 + \frac{1}{\sin(\theta/2)}\right)$$

which falls from infinity at $\theta = 0$ to $D$ at $\theta = 180°$. So if the positive
charge fills a ball of radius $R$, the Coulomb law the alpha has been obeying holds only
while $r_\text{min} > R$, and the angle where the beam first arrives at the surface of
the charge is

$$\sin(\theta_\text{cut}/2) = \frac{1}{2R/D - 1}$$

Past that angle the scattering dies, because a particle *inside* the ball feels only the
fraction of the charge beneath it and cannot be turned far. One equation, both atoms.

In [ ]:
print(f"D, closest approach head-on   {S.D_CLOSEST * 1e15:.1f} fm")
print()
rows = []
for label, radius in (("the diffuse atom", S.R_ATOM),
                      ("a charge ball of 300 fm", 3.0e-13),
                      ("a charge ball of 100 fm", 1.0e-13),
                      ("the gold nucleus", S.R_NUCLEUS)):
    cut = S.cutoff_sine(radius)
    where = "never — the beam does not reach it" if not math.isfinite(cut) else (
        f"{math.degrees(2 * math.asin(min(cut, 1.0))):.3f} deg" if cut < 1.0 else "beyond 180 deg")
    rows.append([label, f"{radius:.3g}", where])
table(rows, headers=("charge distribution", "R (m)", "scattering dies at"))

The diffuse atom's positive charge is so large that the alpha is *already inside it* by a
fortieth of a degree. Everything seen past that angle, on that hypothesis, has to be the
pile-up of many tiny deflections, and there is a hard ceiling on how far that can go.

The model carries this as `lam` $= \log \sin(\theta_\text{cut}/2)$ — a dimensionless
number, so that the information matrix is scale-free and the two hypotheses are two
points on one axis rather than two files.

In [ ]:
print(f"lam, the diffuse atom   {S.LAM_DIFFUSE:8.3f}   (cut-off at "
      f"{math.degrees(2 * math.asin(math.exp(S.LAM_DIFFUSE))):.3f} deg)")
print(f"lam, the hard centre    {S.LAM_HARD:8.3f}   (no cut-off anywhere)")
print(f"they are {(S.LAM_HARD - S.LAM_DIFFUSE) / math.log(10):.1f} decades apart")

## 2 · One mean function, both atoms

The detector is a scintillating screen of solid angle $\omega$ steradians, watched
through a microscope at scattering angle $\theta$ for $t$ seconds. What it counts is

$$\mu = t\,\omega \left[\;
\underbrace{A\,u^{-4}e^{-(u/s)^2}}_{\text{one close encounter}} \;+\;
\underbrace{C\,e^{-\theta^2/2w^2}}_{\text{a thousand glancing ones}} \;+\;
\underbrace{B}_{\text{background}}\;\right],
\qquad u = \sin(\theta/2),\; s = e^{\lambda}$$

Three terms, and only the first one is about the question.

The **single-scattering** term is Rutherford's law, $u^{-4}$, cut off by the size of the
charge. The **core** is the compounding of the thousand-odd glancing encounters every
alpha has crossing the foil; both atoms predict it and predict very nearly the same one,
which is exactly why the small-angle measurements of 1909 settled nothing. The
**background** is what the counter records with the foil taken out.

A data column `foil` is `1` with the foil in the beam and `0` with it out; multiplying the
first two terms by it is how one design can allocate time between the two kinds of run.

In [ ]:
model: ModelSpec = S.model()

print(model.name)
print("  data columns  ", sorted({p for p in ("theta", "omega", "exposure", "foil")}))
print("  free params   ", [p.name for p in free_parameters(model)])
print("  fixed         ", [p.name for p in model.parameters if p.prior.family == "fixed"])
print("  mean dimension", dimension(model.mean), "= sqrt(count), and section 4 says why")
print("  content hash  ", model.content_hash()[:16])

The mean's dimension is $\sqrt{\text{count}}$, not counts, and that is deliberate; section
4 is about it. First, what the two atoms predict.

In [ ]:
grid = np.geomspace(0.2, 179.0, 400)
hard = S.rate_per_steradian(grid, S.HARD_CENTRE)
diffuse = S.rate_per_steradian(grid, S.DIFFUSE)

fig = S.figure("What the two atoms predict, per steradian per second",
               "scattering angle", "counts / sr / s", height=440)
fig.add_trace(go.Scatter(x=grid, y=hard, name="a hard positive centre",
                         line={"color": S.HARD_COLOR, "width": 3}))
fig.add_trace(go.Scatter(x=grid, y=diffuse, name="charge spread through the atom",
                         line={"color": S.DIFFUSE_COLOR, "width": 3}))
fig.add_hline(y=S.BACKGROUND_DENSITY, line={"dash": "dot", "color": S.TRUTH_COLOR},
              annotation_text="the counting background")
fig.update_yaxes(type="log", exponentformat="power", range=[-4, 10])
S.degrees_axis(fig, log=True)
fig.show()

Below about a degree the two curves lie on top of each other: both atoms are showing the
same multiple-scattering core and the difference is a factor, not a fact. Then the diffuse
atom's prediction falls off a cliff — a Gaussian dying against a power law — and by five
degrees it has gone under the background and stays there.

That gap is the whole experiment, and it is worth having the numbers rather than the
picture.

In [ ]:
angles = [0.5, 1.0, 2.0, 3.0, 5.0, 10.0, 30.0, 90.0, 150.0]
h = S.rate_per_steradian(angles, S.HARD_CENTRE)
d = S.rate_per_steradian(angles, S.DIFFUSE)
table(
    [[f"{a:.1f}", f"{x:.4g}", f"{y:.4g}", f"{x / y:.4g}"] for a, x, y in zip(angles, h, d)],
    headers=("angle (deg)", "hard centre", "diffuse atom", "ratio"),
)
print("\n(per steradian per second, background included in both)")

## 3 · What the apparatus can never see

$\sin(\theta_\text{cut}/2) = 1/(2R/D - 1)$ has a denominator, and it goes to zero. Once
$R \le D/2$ the beam never reaches the charge **at any angle**, the cut-off disappears
entirely, and every charge ball smaller than that predicts exactly the same thing.

So the resolving power of this experiment is $D/2$, and $D$ depends only on the beam
energy. No exposure, no aperture, no number of counts moves it. This is the single most
important number to have before spending two hundred hours, and it takes one line.

In [ ]:
floor = S.D_CLOSEST / 2.0
print(f"smallest charge radius this beam can resolve   {floor:.3g} m = {floor * 1e15:.1f} fm")
print(f"the gold nucleus is                            {S.R_NUCLEUS * 1e15:.1f} fm, "
      f"{floor / S.R_NUCLEUS:.0f}x smaller")
print()
table(
    [
        [f"{energy:.2f}", f"{S.D_CLOSEST * S.E_ALPHA / energy / 2 * 1e15:.1f}"]
        for energy in (5.0, 7.68, 20.0, 100.0)
    ],
    headers=("alpha energy (MeV)", "resolution floor (fm)"),
)
print("\nThe experiment returns an upper bound on R, never a value. Buying a better one")
print("means buying a faster alpha, and nothing else will do it.")

In [ ]:
angles_fine = np.geomspace(0.05, 179.0, 400)
r_min = S.D_CLOSEST / 2.0 * (1.0 + 1.0 / np.sin(np.radians(angles_fine) / 2.0))
fig = curve_band(
    angles_fine, r_min * 1e15,
    label="closest approach",
    title="How close the beam ever gets",
    subtitle="r_min(θ) for a 7.68 MeV alpha on gold — the whole experiment lives above this curve",
    x_title="scattering angle (degrees)", y_title="closest approach (fm)",
)
fig.update_xaxes(type="log")
fig.update_yaxes(type="log")
mark_y(fig, S.D_CLOSEST / 2.0 * 1e15, text="the floor: D/2, at 180°", color=CRITICAL)
mark_y(fig, S.R_NUCLEUS * 1e15, text="the gold nucleus")
caption(fig, f"Every trajectory in the experiment sits on this curve, and the curve stops at "
             f"{S.D_CLOSEST / 2 * 1e15:.1f} fm however far the alpha is turned. A charge ball "
             f"smaller than that is invisible to this beam — not poorly measured, invisible — "
             f"which is why the result is a bound and why buying a better one means buying a "
             f"faster alpha.")

Rutherford published exactly this bound in 1911 — that the central charge is confined
within $3.4 \times 10^{-14}$ m — and it is the same arithmetic. Everything that follows
is about making the bound honest, not about making it smaller.

## 4 · Counting is Poisson; the design math is Gaussian

Every number in the next four notebooks comes out of `design.fisher_information`, which
is written for a Gaussian likelihood with a standard deviation. Counts are Poisson, whose
variance is its mean and changes by fourteen orders of magnitude across the angle range.

The bridge is exact, not approximate. `forward` returns $2\sqrt{\mu}$ — the
variance-stabilizing transform of a Poisson count, whose variance is one whatever the
rate — and for any parameter $\psi$,

$$\left(\frac{\partial\, 2\sqrt{\mu}}{\partial \psi}\right)^2 \Big/ 1^2
\;=\; \frac{1}{\mu}\left(\frac{\partial \mu}{\partial \psi}\right)^2$$

and the right-hand side *is* the Poisson Fisher information. So
`fisher_information(..., noise_sd=1.0)` on this surface is the Poisson information, with
nothing to apologize for. That claim is checkable, so it gets checked.

In [ ]:
surface = S.surface()
check = S.station([3.0, 20.0, 60.0], 1e-3, 3600.0)
at = S.truth(math.log(0.30))

anscombe = fisher_information(surface, check, at, 1.0, method="finite").as_array()

# The Poisson information, written out longhand: sum_i (dmu_i/dpsi)(dmu_i/dpsi') / mu_i.
names, step = ("log_a", "lam", "log_c", "log_w", "log_b"), 1e-6
mu = surface.counts(check, at)


def d_mu(name: str) -> np.ndarray:
    up, down = dict(at), dict(at)
    up[name], down[name] = at[name] + step, at[name] - step
    return (surface.counts(check, up) - surface.counts(check, down)) / (2 * step)


poisson = np.array([[float(np.sum(d_mu(a) * d_mu(b) / mu)) for b in names] for a in names])

print(f"largest relative disagreement: {np.max(np.abs(anscombe - poisson) / np.abs(poisson)):.2e}")
print("  (which is the finite-difference step, not a modelling gap)")
table(
    [[n, f"{anscombe[i, i]:.6g}", f"{poisson[i, i]:.6g}"] for i, n in enumerate(names)],
    headers=("parameter", "Anscombe scale", "Poisson, longhand"),
)

In [ ]:
fig = scatter_fit(
    np.abs(poisson).ravel(), np.abs(anscombe).ravel(),
    title="The bridge, checked entry by entry",
    subtitle="every element of the information matrix: Poisson written longhand against the Anscombe surface",
    x_title="|Poisson information|", y_title="|design.fisher_information at noise_sd = 1|",
)
fig.update_xaxes(type="log")
fig.update_yaxes(type="log")
caption(fig, "Twenty-five entries spanning eight orders of magnitude, on the line. The design "
             "machinery in the next four notebooks is Gaussian and the counting is Poisson, "
             "and this is the reason that is a change of variable rather than an approximation.")

## What notebook 1 established

* The two atoms are **one model at two values of `lam`**, four decades apart, and `lam` is
  a monotone function of the radius of the positive charge.
* Below a degree the two predictions differ by a factor; past five degrees they differ by
  a hundred million, and the diffuse atom is under the background.
* The experiment **cannot measure the nuclear radius**. Its floor is $D/2 = 29.6$ fm, set
  by the beam energy alone, and the gold nucleus is four times smaller than that. What
  comes back is an upper bound.
* `fisher_information` on this surface is the exact Poisson information, so every
  precision claim in notebooks 2 to 4 is a statement about counting, not about a Gaussian
  approximation to counting.

What it has *not* established is where to point the detector. A factor of a hundred
million at five degrees looks like it settles everything; notebook 2 is about why it does
not, and what does.